In [ ]:
from roboflow import Roboflow
from ultralytics import YOLO

from getpass import getpass

In [ ]:
rf = Roboflow(api_key=getpass("Enter your Roboflow API key: "))

project = rf.workspace("homeanpr").project("journey-cam-anpr")
version = project.version(2)
dataset = version.download("yolov8")

In [ ]:
yml_path = dataset.location + "/data.yaml"
yml_path

In [ ]:
# Load the model with previous weights
model = YOLO("yolov8n.pt")

In [ ]:
# Train the model on your new dataset
model.train(
    data=yml_path,  # dataset config
    epochs=100,
    imgsz=640,
    batch=16,
    name="ANPR-Model", 
    resume=False  # True only if resuming an interrupted training
)

In [ ]:
# Load the trained model
model = YOLO("best.pt")

# Run evaluation on the test set
metrics = model.val(data=yml_path, split="test")

# Optionally print key metrics
print(metrics.box.map)      # mAP@0.5
print(metrics.box.map50)    # mAP@0.5
print(metrics.box.map75)    # mAP@0.75

In [ ]:
# Run inference on a video file
results = model.predict(
    source=input("Test Path: "),  # path to your video
    save=True,                 # save annotated video to disk
    conf=0.25,                 # confidence threshold
    stream=False               # set to True to show live in a window
)